# <a id='toc1_'></a>[Using gene symbol categories to resolve ambiguous gene symbols](#toc0_)

Gene symbol categories are determined by the relationship the gene symbol has with its associated gene concept. These categories are ranked to show priority and used to assign ambiguous gene symbols to unique gene concepts.

**Table of contents**<a id='toc0_'></a>    
- [Using gene symbol categories to resolve ambiguous gene symbols](#toc1_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
import sys
from pathlib import Path

analysis_dir = Path.cwd().parent

if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import importlib
import re

import civicpy.civic as civic
import gene_ids_in_lit.functions as giilfn
import collision_analysis_shared_variables as casv
import pandas as pd
import functions as fn

importlib.reload(giilfn)
importlib.reload(fn)

<module 'functions' from '/Users/rsaxs014/Desktop/gene-harmony-analysis/analysis/ranked_category_resolver/functions.py'>

In [2]:
ortholog_match_subset_genes_df = pd.read_hdf(
    "../../output/ortholog_match_subset_genes_df.h5", key='df'
    )
previous_symbol_match_subset_genes_df = pd.read_hdf(
    "../../output/previous_symbol_match_subset_genes_df.h5", key='df'
    )
flj_clone_symbol_match_subset_genes_df = pd.read_hdf(
    "../../output/flj_clone_symbol_match_subset_genes_df.h5", key='df'
    )
hgnc_gene_group_match_subset_genes_df = pd.read_hdf(
    "../../output/hgnc_gene_group_match_subset_genes_df.h5", key='df'
    )
phenotype_match_subset_genes_df = pd.read_hdf(
    "../../output/phenotype_match_subset_genes_df.h5", key='df'
    )
protein_mass_match_subset_genes_df = pd.read_hdf(
    "../../output/protein_mass_match_subset_genes_df.h5", key='df'
    )
placeholder_match_subset_genes_df = pd.read_hdf(
    "../../output/placeholder_match_subset_genes_df.h5", key='df'
    )
gene_id_match_subset_genes_df = pd.read_hdf(
    "../../output/gene_id_match_subset_genes_df.h5", key='df'
    )
ncbi_gene_neighbor_match_subset_genes_df = pd.read_hdf(
    "../../output/ncbi_gene_neighbor_match_subset_genes_df.h5", key='df'
    )
ncbi_related_gene_match_subset_genes_df = pd.read_hdf(
    "../../output/ncbi_related_gene_match_subset_genes_df.h5", key='df'
    )
ncbi_gene_interaction_symbol_match_subset_genes_df = pd.read_hdf(
    "../../output/ncbi_gene_interaction_symbol_match_subset_genes_df.h5", key='df'
    )
mgi_withdrawn_symbol_match_subset_genes_df = pd.read_hdf(
    "../../output/mgi_withdrawn_symbol_match_subset_genes_df.h5", key='df'
    )
alt_abbrev_symbol_match_subset_genes_df = pd.read_hdf(
    "../../output/alt_abbrev_symbol_match_subset_genes_df.h5", key='df'
    )

In [3]:
relationship_dfs = {
    "Ortholog Symbols": ortholog_match_subset_genes_df,
    "Previous Symbols": previous_symbol_match_subset_genes_df,
    "Clone Symbols":flj_clone_symbol_match_subset_genes_df,
    "Gene Group Symbols":hgnc_gene_group_match_subset_genes_df,
    "Phenotype Symbols":phenotype_match_subset_genes_df,
    "Protein Mass Symbols":protein_mass_match_subset_genes_df,
    "Placeholder Symbols":placeholder_match_subset_genes_df,
    "Gene ID Symbols":gene_id_match_subset_genes_df,
    "Gene Neighbor Symbols":ncbi_gene_neighbor_match_subset_genes_df,
    "Related Gene Symbols":ncbi_related_gene_match_subset_genes_df,
    "Gene Interaction Symbols":ncbi_gene_interaction_symbol_match_subset_genes_df,
    "Withdrawn Ortholog Symbols":mgi_withdrawn_symbol_match_subset_genes_df,
    "Alternate Abbreviation Symbols":alt_abbrev_symbol_match_subset_genes_df
}

In [4]:
def calculate_relationship_category_ambiguity_score(
    df: pd.DataFrame,
    relationship_name: str
) -> dict[str, str | int | float]:
    """Calculate ambiguity statistics for gene-to-symbol mappings.

    Ambiguity occurs when an alias maps to more than one distinct
    primary gene symbol. Duplicate alias-gene pairs are excluded.

    The ambiguity score is calculated as:

        Ambiguity Score (%) = [Σ(ki - 1) / N] * 100

    where ki is the number of distinct primary genes associated
    with an ambiguous alias, and N is the total number of unique
    alias-gene pairs.

    :param df: DataFrame containing alias symbols ('alias_symbol' or
        'gene_symbol') and primary gene symbols ('primary_gene_symbol').
    :type df: pd.DataFrame
    :param relationship_name: Name of the gene relationship being evaluated.
    :type relationship_name: str
    :return: Dictionary containing relationship name, total unique
        alias-gene pairs, number of ambiguous aliases, affected pairs,
        sigma numerator, and ambiguity score (%).
    :rtype: dict[str, str | int | float]
    """
    alias_column = (
        "alias_symbol" if "alias_symbol" in df.columns
        else "gene_symbol"
    )

    pairs = (
        df[[alias_column, "primary_gene_symbol"]]
        .dropna()
        .drop_duplicates()
    )

    gene_counts = pairs.groupby(alias_column)["primary_gene_symbol"].nunique()
    ambiguous_counts = gene_counts[gene_counts > 1]

    total_pairs = len(pairs)
    sigma_numerator = int((ambiguous_counts - 1).sum())

    ambiguity_score = (
        sigma_numerator / total_pairs * 100
        if total_pairs > 0 else float("nan")
    )

    return {
        "Relationship": relationship_name,
        "Total Annotated Alias–Gene Pairs": total_pairs,
        "Aliases Mapping to Multiple Genes": len(ambiguous_counts),
        "Affected Alias–Gene Pairs": int(ambiguous_counts.sum()),
        "Sigma Numerator": sigma_numerator,
        "Ambiguity Score (%)": round(ambiguity_score, 2)
    }

## Calculate the ambiguity score of each relationship category

In [ ]:
ambiguity_table = pd.DataFrame([
    calculate_relationship_category_ambiguity_score(df, relationship)
    for relationship, df in relationship_dfs.items()
]).sort_values(by="Ambiguity Score (%)", ascending=True).reset_index(drop=True)

ambiguity_table

,Relationship,Total Annotated Alias–Gene Pairs,Aliases Mapping to Multiple Genes,Affected Alias–Gene Pairs,Sigma Numerator,Ambiguity Score (%)
0,Ortholog Symbols,3070,67,191,124,4.04
1,Previous Symbols,19805,189,428,239,1.21
2,Clone Symbols,3574,19,38,19,0.53
3,Gene Group Symbols,12916,355,1059,704,5.45
4,Phenotype Symbols,6892,62,132,70,1.02
5,Protein Mass Symbols,123,11,24,13,10.57
6,Placeholder Symbols,9738,98,201,103,1.06
7,Gene ID Symbols,4664,2,4,2,0.04
8,Gene Neighbor Symbols,278,18,51,33,11.87
9,Related Gene Symbols,130,6,16,10,7.69


## Prepare Ranked Relationship List for use in Gene JAR

In [6]:
relationship_to_category = {
    "Ortholog Symbols": "ORTHOLOG",
    "Previous Symbols": "PREVIOUS",
    "Clone Symbols": "CLONE_NAME",
    "Gene Group Symbols": "GENE_GROUP",
    "Phenotype Symbols": "PHENOTYPE",
    "Protein Mass Symbols": "PROTEIN_MASS",
    "Placeholder Symbols": "PLACEHOLDER",
    "Gene ID Symbols": "GENE_IDENTIFIER",
    "Gene Neighbor Symbols": "GENE_NEIGHBOR",
    "Related Gene Symbols": "RELATED_GENE",
    "Gene Interaction Symbols": "GENE_INTERACTION",
    "Withdrawn Ortholog Symbols": "WITHDRAWN_ORTHOLOG",
    "Alternate Abbreviation Symbols": "ALTERNATE_ABBREVIATION",
}

sorted_relationships = ambiguity_table.sort_values(
    by="Ambiguity Score (%)",
    ascending=True,
    kind="stable"
)

print("RANK_ORDER: ClassVar[list[SymbolCategory]] = [")
print("    SymbolCategory.PRIMARY,")

for relationship in sorted_relationships["Relationship"]:
    category = relationship_to_category[relationship]
    print(f"    SymbolCategory.{category},")

print("]")

RANK_ORDER: ClassVar[list[SymbolCategory]] = [
    SymbolCategory.PRIMARY,
    SymbolCategory.GENE_IDENTIFIER,
    SymbolCategory.WITHDRAWN_ORTHOLOG,
    SymbolCategory.CLONE_NAME,
    SymbolCategory.PHENOTYPE,
    SymbolCategory.PLACEHOLDER,
    SymbolCategory.PREVIOUS,
    SymbolCategory.ALTERNATE_ABBREVIATION,
    SymbolCategory.ORTHOLOG,
    SymbolCategory.GENE_GROUP,
    SymbolCategory.RELATED_GENE,
    SymbolCategory.PROTEIN_MASS,
    SymbolCategory.GENE_INTERACTION,
    SymbolCategory.GENE_NEIGHBOR,
]
